# SLM QLoRA Evaluation — Multi-Adapter Runner

Refactored to automatically evaluate all 15 trained adapters  
(5 supervision strategies × 3 seeds) without manual path changes.

**Architecture:**
```
RUN REGISTRY
    ↓
For each adapter:
    Load fresh Qwen2-1.5B
    ↓
    Attach exactly one adapter
    ↓
    Run existing evaluation pipeline
    ↓
    Save raw predictions + parsed predictions + metrics
    ↓
    Unload adapter/model + Clear GPU
    ↓
After all 15: Aggregate mean ± std
```

## Cell 0 — Install Dependencies

In [ ]:
!pip install torch torchvision torchaudio torchao --index-url https://download.pytorch.org/whl/cu121

!pip install -q transformers==5.6.2
!pip install -q accelerate==1.13.0
!pip install -q peft>=0.19.1
!pip install -q bitsandbytes>=0.49.2
!pip install -q trl==0.19.0
!pip install -q "datasets<4.0.0"
!pip install -q evaluate>=0.4.6
!pip install -q scikit-learn stanza

## Cell 1 — Environment Configuration

> **Only change `adapter_root` and `results_root` when moving between machines.**

In [ ]:
# ============================================================
# ENVIRONMENT CONFIGURATION
# Only these paths need to change between machines.
# ============================================================
CONFIG = {
    "base_model_name": "Qwen/Qwen2-1.5B",
    "adapter_root": "Riset_QLoRA/runs",     # <-- CHANGE THIS
    "results_root": "evaluation_results/",  # <-- CHANGE THIS

    # Evaluation params
    "batch_size": 1,
    "eval_limit": 150,  # set to None to evaluate all samples
}

# ============================================================
# DEBUG / CONTROL FLAGS
# ============================================================
# Set DRY_RUN = True to validate paths without loading models.
DRY_RUN = False

# Set TEST_ONE_RUN = True to evaluate a single adapter first.
TEST_ONE_RUN = False
TEST_RUN_NAME = "LiSA_seed42"  # which run to test

print("✅ Configuration loaded")
print(f"   Base model : {CONFIG['base_model_name']}")
print(f"   Adapter root: {CONFIG['adapter_root']}")
print(f"   Results root: {CONFIG['results_root']}")
print(f"   Batch size  : {CONFIG['batch_size']}")
print(f"   Eval limit  : {CONFIG['eval_limit']}")
print(f"   DRY_RUN     : {DRY_RUN}")
print(f"   TEST_ONE_RUN: {TEST_ONE_RUN} ({TEST_RUN_NAME if TEST_ONE_RUN else 'N/A'})")

## Cell 2 — Experiment Definition & Adapter Registry

15 runs = 5 strategies × 3 seeds.  
Adapter paths are constructed automatically from `CONFIG["adapter_root"]`.

In [ ]:
import os

# ============================================================
# EXPERIMENT DEFINITION
# Maps strategy name -> linguistic_mode used during training.
# Do NOT include NORM_MORPH or vocabulary-adaptation experiments.
# ============================================================
EXPERIMENTS = {
    "QLoRA":        "NONE",
    "Normalization": "NORMALIZATION_ONLY",
    "Register":     "REGISTER_ONLY",
    "Morphology":   "MORPH_ONLY",
    "LiSA":         "FULL_PIPELINE",
}

SEEDS = [42, 123, 456]

# ============================================================
# BUILD RUN REGISTRY
# Each run has a unique name, strategy, linguistic_mode,
# seed, and adapter_path derived from CONFIG.
# ============================================================
RUNS = []

for strategy_name, linguistic_mode in EXPERIMENTS.items():
    for seed in SEEDS:
        run_name = f"{strategy_name}_seed{seed}"
        RUNS.append({
            "run_name":       run_name,
            "strategy":       strategy_name,
            "linguistic_mode": linguistic_mode,
            "seed":           seed,
            "adapter_path":   os.path.join(CONFIG["adapter_root"], run_name),
        })

print(f"Run registry built: {len(RUNS)} runs total")
print()
for i, run in enumerate(RUNS, 1):
    print(f"  {i:2d}. {run['run_name']:30s}  mode={run['linguistic_mode']}")

## Cell 3 — Import Libraries

In [ ]:
import torch
import gc
import os
import json
import re
import string
import collections
import time
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from sklearn.metrics import accuracy_score, f1_score
from tqdm import tqdm
from datasets import load_dataset

print("✅ Libraries imported")

## Cell 4 — GPU Check

In [ ]:
# Force PyTorch to use only the first GPU
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

print("=" * 50)
print("GPU CHECK")
print("=" * 50)

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        vram_gb = props.total_memory / (1024**3)
        print(f"GPU {i}: {props.name} — {vram_gb:.1f} GB VRAM")
    print(f"\nTotal GPU: {torch.cuda.device_count()}")
    print(f"CUDA version: {torch.version.cuda}")
else:
    print("❌ GPU not found! Make sure Kaggle Accelerator is enabled.")
    print("Go to: Settings → Accelerator → GPU T4 x2")

## Cell 5 — Load Evaluation Datasets (once, shared across all adapters)

The evaluation dataset must be **exactly the same** for all 15 runs.  
Datasets are loaded here and never reloaded during the evaluation loop.

In [ ]:
# ============================================================
# LOAD EVALUATION DATASETS
# Loaded once and reused across all 15 adapter evaluations.
# eval_limit caps the number of examples per task.
# ============================================================
print("Loading evaluation datasets...")
eval_tasks = {}

try:
    ds = load_dataset("haryoaw/COPAL", split="test", trust_remote_code=True)
    if CONFIG["eval_limit"] is not None:
        ds = ds.select(range(min(CONFIG["eval_limit"], len(ds))))
    eval_tasks["copal"] = ds
    print(f"✅ COPAL-ID: {len(eval_tasks['copal'])} samples")
except Exception as e:
    print(f"❌ COPAL-ID failed: {e}")

try:
    ds = load_dataset("afaji/indonli", split="test_expert", trust_remote_code=True)
    if CONFIG["eval_limit"] is not None:
        ds = ds.select(range(min(CONFIG["eval_limit"], len(ds))))
    eval_tasks["indonli"] = ds
    print(f"✅ IndoNLI: {len(eval_tasks['indonli'])} samples")
except Exception as e:
    print(f"❌ IndoNLI failed: {e}")

try:
    ds = load_dataset("indolem/IndoCulture", split="test", trust_remote_code=True)
    if CONFIG["eval_limit"] is not None:
        ds = ds.select(range(min(CONFIG["eval_limit"], len(ds))))
    eval_tasks["indoculture"] = ds
    print(f"✅ IndoCulture: {len(eval_tasks['indoculture'])} samples")
except Exception as e:
    print(f"❌ IndoCulture failed: {e}")

print()
print(f"Total tasks loaded: {list(eval_tasks.keys())}")

## Cell 6 — Evaluation Functions (Canonical, Preserved from Original Notebook)

> **NOTE**: These functions are preserved exactly from the original notebook.  
> The canonical evaluation function is `evaluate_batched_final`.  
> `evaluate_batched` and `evaluate_batched_c3` are retained as **LEGACY / UNUSED** references only.

The `format_prompt` function has been updated to accept `linguistic_mode`  
(e.g., `NONE`, `NORMALIZATION_ONLY`, `REGISTER_ONLY`, `MORPH_ONLY`, `FULL_PIPELINE`)  
instead of the old `condition_name` strings (C3_Linguistic, C4_Vocabulary, etc.).  
The logic is **identical**: any mode that is not `NONE` adds the `[FORMAL]` tag prefix.

In [ ]:
# ============================================================
# LABEL MAPPING
# Preserved exactly from original notebook.
# ============================================================

def map_hybrid_label(task_name, sample):
    """Menyamakan format label dataset (Mengatasi tipe data string/integer yang bentrok)"""

    # 2. Untuk dataset selain QA
    raw_label = str(sample.get("label", sample.get("answer", sample.get("target", "-1")))).strip().upper()

    if task_name == "copal":
        # 0 -> A, 1 -> B
        return "A" if raw_label == "0" else "B"

    elif task_name == "indoculture":
        label_map = {"0": "A", "1": "B", "2": "C"}
        return raw_label if raw_label in ["A", "B", "C"] else label_map.get(raw_label, "UNKNOWN")

    elif task_name == "indonli":
        label_map = {"0": "SESUAI", "1": "NETRAL", "2": "BERTENTANGAN"}
        return label_map.get(raw_label, "UNKNOWN")

    return raw_label


# ============================================================
# OUTPUT PARSING
# Preserved exactly from original notebook.
# ============================================================

def extract_hybrid_prediction(task_name, gen_text):
    """
    Mengekstrak label jawaban secara pemaaf menggunakan Regex,
    mengakomodasi model finetuned yang mengalami Excessive Verbosity.
    """
    gen_text = str(gen_text).strip().upper()

    if task_name in ["copal", "indoculture"]:
        # 1. Deteksi Jebakan "A": Cari huruf A, B, C di awal kalimat yang diikuti titik, spasi, atau kurung
        match = re.search(r'^([A-C])[\.\s\(\:]', gen_text)
        if match:
            return match.group(1)

        # 2. Jika model hanya menjawab murni 1 huruf
        if len(gen_text) == 1 and gen_text in ['A', 'B', 'C']:
            return gen_text

        # 3. Cari pola kalimat penjabaran panjang
        for label in ["A", "B", "C"]:
            if f"JAWABAN: {label}" in gen_text or f"ADALAH {label}" in gen_text or f"PILIHAN {label}" in gen_text:
                return label

    elif task_name == "indonli":
        # Sinkronisasi dengan map_hybrid_label ("SESUAI", "NETRAL", "BERTENTANGAN")
        # Menangkap variasi bahasa Inggris dari instruksi dan menerjemahkannya
        if "SESUAI" in gen_text or "ENTAILMENT" in gen_text: return "SESUAI"
        if "BERTENTANGAN" in gen_text or "CONTRADICTION" in gen_text: return "BERTENTANGAN"
        if "NETRAL" in gen_text or "NEUTRAL" in gen_text: return "NETRAL"

    return "UNKNOWN"


print("✅ Label mapping and output parsing functions loaded")

In [ ]:
# ============================================================
# PROMPT CONSTRUCTION
# Updated to use linguistic_mode instead of condition_name.
# Logic is identical: NONE -> no tag; all other modes -> [FORMAL] tag.
# ============================================================

def format_prompt(sample, task_name, linguistic_mode):
    """
    Membangun isi konten untuk "user_msg".
    TIDAK menggunakan ### Respons: karena add_generation_prompt=True sudah mengurusnya.

    linguistic_mode: one of NONE | NORMALIZATION_ONLY | REGISTER_ONLY | MORPH_ONLY | FULL_PIPELINE
    - NONE          -> no [FORMAL] tag (baseline QLoRA)
    - all others    -> [FORMAL] tag prefix (linguistic supervision)
    """
    # Determine tag prefix:
    # Any mode except NONE uses the [FORMAL] tag, matching original C3/C4 behaviour.
    tag_prefix = "" if linguistic_mode == "NONE" else "[FORMAL]\n"

    # 1. INDONLI
    if task_name == "indonli":
        p, h = sample.get("premise", "").strip(), sample.get("hypothesis", "").strip()
        return (
            "Diberikan sebuah Premis dan Hipotesis. Tentukan hubungan logis di antara keduanya.\n"
            "Jawab HANYA dengan satu kata: ENTAILMENT, NEUTRAL, atau CONTRADICTION.\n\n"
            f"Premis: {p}\n"
            f"Hipotesis: {h}"
        )

    # 2. COPAL
    elif task_name == "copal":
        q_type = str(sample.get("question", "")).lower()
        pertanyaan = ("Apa penyebab dari situasi tersebut?" if q_type == "cause" else
                      "Apa akibat dari situasi tersebut?"   if q_type == "effect" else
                      "Manakah pilihan yang paling masuk akal?")
        inst_text = (
            f"Situasi: {sample['premise']}\n"
            f"Pertanyaan: {pertanyaan}\n"
            f"A. {sample['choice1']}\n"
            f"B. {sample['choice2']}\n"
            "Jawaban (A atau B):"
        )
        return f"{tag_prefix}Instruksi:\n{inst_text}" if tag_prefix else f"Instruksi:\n{inst_text}"

    # 3. INDOCULTURE
    elif task_name == "indoculture":
        options_list = sample.get("options", [])
        options_text = "\n".join(options_list) if isinstance(options_list, list) else str(options_list)
        inst_text = (
            "Bacalah pertanyaan berikut dan pilih jawaban yang paling tepat berdasarkan konteks budaya Indonesia.\n\n"
            f"Konteks: {sample['context']}\n"
            f"Pilihan:\n{options_text}\n\n"
            "Jawaban (A, B, atau C):"
        )
        return f"{tag_prefix}Instruksi:\n{inst_text}" if tag_prefix else f"Instruksi:\n{inst_text}"

    return ""


print("✅ format_prompt loaded (linguistic_mode-aware)")

In [ ]:
# ============================================================
# CANONICAL EVALUATION FUNCTION
# This is evaluate_batched_final — the function that produced
# the reported paper results. Preserved exactly.
# linguistic_mode is now an explicit argument.
# ============================================================

def evaluate_batched_final(model, tokenizer, dataset, task_name, linguistic_mode, batch_size=8):
    model.eval()
    results = []

    for i in tqdm(range(0, len(dataset), batch_size), desc=f"Eval {task_name} ({linguistic_mode})"):
        batch = dataset.select(range(i, min(i + batch_size, len(dataset))))
        prompts = []

        for s in batch:
            # 1. Dapatkan string konten user
            p_content = format_prompt(s, task_name, linguistic_mode)

            # 2. BUNGKUS DENGAN CHATML (Sama persis seperti saat training C3/C4)
            msg = [{"role": "user", "content": p_content}]

            # add_generation_prompt=True otomatis menambahkan <|im_start|>assistant\n
            full_prompt = tokenizer.apply_chat_template(msg, tokenize=False, add_generation_prompt=True)
            prompts.append(full_prompt)

        # --- Proses Inferensi ---
        inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True).to(model.device)

        with torch.no_grad():
            start_inf = time.perf_counter()
            outputs = model.generate(**inputs, max_new_tokens=10, do_sample=False, pad_token_id=tokenizer.eos_token_id)
            end_inf = time.perf_counter()

        for j, out in enumerate(outputs):
            gen_text = tokenizer.decode(out[inputs["input_ids"].shape[1]:], skip_special_tokens=True).upper()
            results.append({"pred": gen_text, "latency": (end_inf - start_inf)/len(batch)})

    return results


# ============================================================
# LEGACY EVALUATION FUNCTIONS — NOT USED IN PAPER RESULTS
# Retained for historical reference only.
# Do not call these from the main evaluation loop.
# ============================================================

def _LEGACY_evaluate_batched(model, tokenizer, dataset, task_name, batch_size=8):
    """LEGACY — early version. NOT used for paper results. Use evaluate_batched_final."""
    model.eval()
    results = []

    for i in tqdm(range(0, len(dataset), batch_size), desc=f"Eval {task_name}"):
        batch = dataset.select(range(i, min(i + batch_size, len(dataset))))
        prompts = []

        for s in batch:
            if task_name == "copal":
                q_type = str(s.get("question", "")).lower()
                if q_type == "cause":
                    pertanyaan = "Apa penyebab dari situasi tersebut?"
                elif q_type == "effect":
                    pertanyaan = "Apa akibat dari situasi tersebut?"
                else:
                    pertanyaan = "Manakah pilihan yang paling masuk akal?"
                p = (
                    f"Situasi: {s['premise']}\n"
                    f"Pertanyaan: {pertanyaan}\n"
                    f"A. {s['choice1']}\n"
                    f"B. {s['choice2']}\n"
                    "Jawaban (A/B):"
                )
            elif task_name == "indoculture":
                options_list = s.get("options", [])
                if isinstance(options_list, list):
                    options_text = "\n".join(options_list)
                else:
                    options_text = str(options_list)
                p = (
                    f"Konteks: {s['context']}\n"
                    f"Pilihan:\n{options_text}\n\n"
                    "Jawaban (A/B/C):"
                )
            elif task_name == "indonli":
                p = f"Premis: {s['premise']}\nHipotesis: {s['hypothesis']}\nHubungan (Entailment/Contradiction/Neutral):"
            else:
                p = f"Tentukan sentimen kalimat berikut (positif/negatif/netral):\n'{s['text']}'\nSentimen:"

            msg = [{"role": "user", "content": p}]
            prompts.append(tokenizer.apply_chat_template(msg, tokenize=False, add_generation_prompt=True))

        inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True).to(model.device)

        with torch.no_grad():
            start_inf = time.perf_counter()
            outputs = model.generate(**inputs, max_new_tokens=10, do_sample=False, pad_token_id=tokenizer.eos_token_id)
            end_inf = time.perf_counter()

        for j, out in enumerate(outputs):
            gen_text = tokenizer.decode(out[inputs["input_ids"].shape[1]:], skip_special_tokens=True).upper()
            results.append({"pred": gen_text, "latency": (end_inf - start_inf)/len(batch)})

    return results


print("✅ Canonical evaluate_batched_final loaded")
print("ℹ️  Legacy functions prefixed with _LEGACY_ (not called in evaluation loop)")

## Cell 7 — Adapter Validation & Dry Run

In [ ]:
# ============================================================
# ADAPTER VALIDATION
# Check that expected PEFT/LoRA files exist before evaluation.
# ============================================================
REQUIRED_ADAPTER_FILES = ["adapter_config.json", "adapter_model.safetensors"]

def validate_adapter(adapter_path, run_name):
    """Raise FileNotFoundError if the adapter directory or required files are missing."""
    if not os.path.isdir(adapter_path):
        raise FileNotFoundError(
            f"[{run_name}] Adapter directory not found: {adapter_path}"
        )
    for fname in REQUIRED_ADAPTER_FILES:
        if not os.path.isfile(os.path.join(adapter_path, fname)):
            raise FileNotFoundError(
                f"[{run_name}] Required file '{fname}' missing in: {adapter_path}"
            )


# ============================================================
# DRY RUN MODE
# Prints planned evaluations and adapter paths without loading models.
# ============================================================
if DRY_RUN:
    print("=" * 60)
    print("DRY RUN — No models will be loaded.")
    print("=" * 60)
    print("Planned evaluations:\n")

    all_valid = True
    for i, run in enumerate(RUNS, 1):
        path_exists = os.path.isdir(run["adapter_path"])
        status = "✅" if path_exists else "❌ MISSING"
        print(f"  {i:2d}. {run['run_name']:30s}  mode={run['linguistic_mode']:20s}  {status}")
        print(f"       Path: {run['adapter_path']}")
        if not path_exists:
            all_valid = False

    print()
    if all_valid:
        print("✅ All 15 adapter paths found. Set DRY_RUN = False to proceed.")
    else:
        print("❌ Some adapter paths are missing. Fix them before setting DRY_RUN = False.")
else:
    print("DRY_RUN is False — proceeding to evaluation.")
    print("(Set DRY_RUN = True in Cell 1 to validate paths first.)")

## Cell 8 — `evaluate_one_run`: Core Evaluation Function

In [ ]:
# ============================================================
# 4-BIT QUANTIZATION CONFIG (shared, defined once)
# ============================================================
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)


def evaluate_one_run(
    run_name,
    strategy,
    linguistic_mode,
    seed,
    adapter_path,
):
    """
    Evaluate a single adapter on all benchmarks.

    Steps:
    1. Validate adapter files.
    2. Check for existing COMPLETED marker (skip if present).
    3. Load a fresh Qwen2-1.5B base model.
    4. Attach exactly one adapter.
    5. Run evaluate_batched_final on each benchmark.
    6. Save raw predictions + parsed predictions + metrics.
    7. Write COMPLETED marker.
    8. Delete model/tokenizer and clear GPU memory.

    Returns a dict of per-benchmark metrics, or raises on failure.
    """
    run_dir = os.path.join(CONFIG["results_root"], run_name)
    completed_marker = os.path.join(run_dir, "COMPLETED")

    # ----------------------------------------------------------
    # OVERWRITE PROTECTION
    # ----------------------------------------------------------
    if os.path.isfile(completed_marker):
        print(f"⏭️  Skipping already completed run: {run_name}")
        # Load and return existing metrics
        metrics_path = os.path.join(run_dir, "metrics.json")
        if os.path.isfile(metrics_path):
            with open(metrics_path, "r", encoding="utf-8") as f:
                saved = json.load(f)
            return saved.get("benchmark_metrics", {})
        return {}

    print()
    print("=" * 60)
    print(f"🔥 EVALUATING: {run_name}")
    print(f"   Strategy      : {strategy}")
    print(f"   Linguistic mode: {linguistic_mode}")
    print(f"   Seed          : {seed}")
    print(f"   Adapter path  : {adapter_path}")
    print("=" * 60)

    # ----------------------------------------------------------
    # STEP 1: Validate adapter
    # ----------------------------------------------------------
    validate_adapter(adapter_path, run_name)

    os.makedirs(run_dir, exist_ok=True)

    # ----------------------------------------------------------
    # STEP 2: Load a FRESH base model + tokenizer
    # MANDATORY: Never reuse base model across adapters.
    # ----------------------------------------------------------
    print(f"⏳ Loading tokenizer from {adapter_path}...")
    tokenizer = AutoTokenizer.from_pretrained(adapter_path)
    tokenizer.padding_side = "left"

    print(f"⏳ Loading fresh {CONFIG['base_model_name']} (4-bit)...")
    base_model = AutoModelForCausalLM.from_pretrained(
        CONFIG["base_model_name"],
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )

    # Resize token embeddings (safe for all conditions)
    base_model.resize_token_embeddings(len(tokenizer))

    # ----------------------------------------------------------
    # STEP 3: Attach exactly ONE adapter
    # ----------------------------------------------------------
    print("⏳ Attaching LoRA adapter...")
    model = PeftModel.from_pretrained(base_model, adapter_path)
    print("✅ Model ready for evaluation!")

    # ----------------------------------------------------------
    # STEP 4: Save run metadata
    # ----------------------------------------------------------
    run_metadata = {
        "run_name":        run_name,
        "strategy":        strategy,
        "linguistic_mode": linguistic_mode,
        "seed":            seed,
        "adapter_path":    adapter_path,
        "base_model":      CONFIG["base_model_name"],
        "eval_config": {
            "batch_size":          CONFIG["batch_size"],
            "eval_limit":          CONFIG["eval_limit"],
            "max_new_tokens":      10,
            "do_sample":           False,
            "quantization":        "4-bit NF4",
            "prompt_version":      "v2_format_prompt_linguistic_mode",
        }
    }

    # ----------------------------------------------------------
    # STEP 5: Evaluate all benchmarks
    # ----------------------------------------------------------
    benchmark_metrics = {}

    for task_name, dataset in eval_tasks.items():
        print(f"\n   Evaluating {task_name.upper()} ({len(dataset)} samples)...")

        # Run canonical evaluation function (unchanged from original)
        raw_results = evaluate_batched_final(
            model=model,
            tokenizer=tokenizer,
            dataset=dataset,
            task_name=task_name,
            linguistic_mode=linguistic_mode,
            batch_size=CONFIG["batch_size"],
        )

        # ---- Parse predictions and compute metrics ----
        y_true, y_pred = [], []
        format_errors = 0
        prediction_records = []  # raw + parsed + validity

        for i, res in enumerate(raw_results):
            raw_output       = res["pred"]
            parsed_prediction = extract_hybrid_prediction(task_name, raw_output)
            ground_truth     = map_hybrid_label(task_name, dataset[i])
            is_valid         = parsed_prediction != "UNKNOWN"
            is_correct       = parsed_prediction == ground_truth

            if not is_valid:
                format_errors += 1

            y_true.append(ground_truth)
            y_pred.append(parsed_prediction)

            prediction_records.append({
                "id":               i,
                "ground_truth":     ground_truth,
                "raw_output":       raw_output,
                "parsed_prediction": parsed_prediction,
                "valid":            is_valid,
                "correct":          is_correct,
                "latency":          res["latency"],
            })

        # Compute metrics (preserved from original)
        acc         = accuracy_score(y_true, y_pred)
        unique_labels = list(set(y_true))
        macro_f1    = f1_score(y_true, y_pred, labels=unique_labels, average="macro", zero_division=0)

        valid_count   = len(prediction_records) - format_errors
        invalid_count = format_errors
        total_count   = len(prediction_records)

        print(f"      📊 Accuracy: {acc*100:.2f}% | F1: {macro_f1*100:.2f}% | "
              f"Invalid: {invalid_count}/{total_count}")

        benchmark_metrics[task_name] = {
            "accuracy":     round(acc * 100, 4),
            "macro_f1":     round(macro_f1 * 100, 4),
            "valid_count":  valid_count,
            "invalid_count": invalid_count,
            "total_count":  total_count,
        }

        # ---- Save per-benchmark predictions ----
        predictions_path = os.path.join(run_dir, f"{task_name}_predictions.json")
        with open(predictions_path, "w", encoding="utf-8") as f:
            json.dump(prediction_records, f, ensure_ascii=False, indent=2)

    # ----------------------------------------------------------
    # STEP 6: Save metrics.json
    # ----------------------------------------------------------
    metrics_out = {
        "run_metadata":    run_metadata,
        "benchmark_metrics": benchmark_metrics,
    }
    metrics_path = os.path.join(run_dir, "metrics.json")
    with open(metrics_path, "w", encoding="utf-8") as f:
        json.dump(metrics_out, f, ensure_ascii=False, indent=2)

    # ----------------------------------------------------------
    # STEP 7: Write COMPLETED marker
    # Only written after all benchmarks finish successfully.
    # ----------------------------------------------------------
    with open(completed_marker, "w") as f:
        f.write("COMPLETED\n")

    print(f"\n✅ {run_name} complete. Results saved to: {run_dir}")

    # ----------------------------------------------------------
    # STEP 8: GPU CLEANUP — MANDATORY between adapters
    # ----------------------------------------------------------
    print("🧹 Cleaning GPU VRAM...")
    del model
    del base_model
    del tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    print("✅ VRAM cleared. Ready for next adapter.")

    return benchmark_metrics


print("✅ evaluate_one_run defined")

## Cell 9 — Aggregation Function

In [ ]:
def aggregate_results():
    """
    Read all completed per-run metrics.json files and produce:
    - all_results.csv     : one row per adapter/seed combination
    - aggregate_results.csv: mean ± std across seeds per strategy
    """
    results_root = CONFIG["results_root"]
    rows = []

    for run in RUNS:
        run_name   = run["run_name"]
        metrics_path = os.path.join(results_root, run_name, "metrics.json")

        if not os.path.isfile(metrics_path):
            print(f"⚠️  No metrics.json for {run_name} — skipping from aggregate.")
            continue

        with open(metrics_path, "r", encoding="utf-8") as f:
            saved = json.load(f)

        bm = saved.get("benchmark_metrics", {})

        row = {
            "strategy":              run["strategy"],
            "seed":                  run["seed"],
            "run_name":              run_name,
            "indonli_accuracy":      bm.get("indonli",     {}).get("accuracy",  None),
            "indonli_macro_f1":      bm.get("indonli",     {}).get("macro_f1",  None),
            "copal_accuracy":        bm.get("copal",       {}).get("accuracy",  None),
            "copal_macro_f1":        bm.get("copal",       {}).get("macro_f1",  None),
            "indoculture_accuracy":  bm.get("indoculture", {}).get("accuracy",  None),
            "indoculture_macro_f1":  bm.get("indoculture", {}).get("macro_f1",  None),
        }
        rows.append(row)

    if not rows:
        print("❌ No completed runs found. Nothing to aggregate.")
        return

    df_all = pd.DataFrame(rows)
    all_csv_path = os.path.join(results_root, "all_results.csv")
    df_all.to_csv(all_csv_path, index=False)
    print(f"✅ Saved: {all_csv_path}  ({len(df_all)} rows)")

    # ---- Aggregate: mean ± std per strategy ----
    metric_cols = [
        "indonli_accuracy", "indonli_macro_f1",
        "copal_accuracy",   "copal_macro_f1",
        "indoculture_accuracy", "indoculture_macro_f1",
    ]

    agg_rows = []
    for strategy in EXPERIMENTS.keys():
        df_s = df_all[df_all["strategy"] == strategy]
        if df_s.empty:
            continue
        agg_row = {"strategy": strategy, "n_seeds": len(df_s)}
        for col in metric_cols:
            vals = df_s[col].dropna()
            agg_row[f"{col}_mean"] = round(vals.mean(), 4) if len(vals) > 0 else None
            agg_row[f"{col}_std"]  = round(vals.std(ddof=1), 4) if len(vals) > 1 else None
        agg_rows.append(agg_row)

    df_agg = pd.DataFrame(agg_rows)
    agg_csv_path = os.path.join(results_root, "aggregate_results.csv")
    df_agg.to_csv(agg_csv_path, index=False)
    print(f"✅ Saved: {agg_csv_path}  ({len(df_agg)} rows)")

    print()
    print("=" * 80)
    print("AGGREGATE RESULTS (mean ± std across seeds)")
    print("=" * 80)

    display_cols = ["strategy", "n_seeds"]
    for col in metric_cols:
        display_cols += [f"{col}_mean", f"{col}_std"]

    print(df_agg[display_cols].to_string(index=False))

    return df_all, df_agg


print("✅ aggregate_results defined")

## Cell 10 — 🚀 Main Evaluation Loop (One-Click Execution)

**How to use:**
1. Set `adapter_root` and `results_root` in Cell 1.
2. Set `DRY_RUN = True` in Cell 1, run all cells to validate paths.
3. Set `DRY_RUN = False` in Cell 1 and click **Run All**.

**Optional single-run debug mode:**  
Set `TEST_ONE_RUN = True` and `TEST_RUN_NAME = "LiSA_seed42"` in Cell 1.

In [ ]:
# ============================================================
# MAIN EVALUATION LOOP
# ============================================================
if DRY_RUN:
    print("DRY_RUN = True — skipping model evaluation.")
    print("Set DRY_RUN = False in Cell 1 to begin evaluation.")
else:
    os.makedirs(CONFIG["results_root"], exist_ok=True)

    # Determine which runs to execute
    if TEST_ONE_RUN:
        runs_to_execute = [r for r in RUNS if r["run_name"] == TEST_RUN_NAME]
        if not runs_to_execute:
            raise ValueError(f"TEST_RUN_NAME '{TEST_RUN_NAME}' not found in RUNS registry.")
        print(f"🔬 TEST_ONE_RUN mode: evaluating only {TEST_RUN_NAME}")
    else:
        runs_to_execute = RUNS
        print(f"🔥 Starting full evaluation: {len(runs_to_execute)} runs")

    successful_runs = []
    failed_runs     = []

    for run in runs_to_execute:
        try:
            evaluate_one_run(**run)
            successful_runs.append(run["run_name"])
        except Exception as e:
            failed_runs.append(run["run_name"])
            print(f"\n❌ {run['run_name']} FAILED: {e}")
            # Attempt cleanup even on failure
            try:
                del model
            except NameError:
                pass
            try:
                del base_model
            except NameError:
                pass
            try:
                del tokenizer
            except NameError:
                pass
            gc.collect()
            torch.cuda.empty_cache()

    # ---- Summary ----
    print()
    print("=" * 60)
    print("EVALUATION SUMMARY")
    print("=" * 60)
    print(f"Successful: {len(successful_runs)} / {len(runs_to_execute)}")
    if failed_runs:
        print("Failed:")
        for name in failed_runs:
            print(f"  - {name}")
    else:
        print("All runs completed successfully! ✅")

## Cell 11 — Generate Master Results (all_results.csv + aggregate_results.csv)

In [ ]:
if not DRY_RUN:
    aggregate_results()
else:
    print("DRY_RUN = True — skipping aggregation.")